# UndertriAI — GRPO Training (Kaggle/Colab)

3-level curriculum training for bail assessment using Qwen2.5-7B-Instruct + Unsloth GRPO.

| Level | Focus | Episodes | Steps |
|-------|-------|----------|-------|
| **Format** | XML structure, think blocks | 400 | 80 |
| **Reasoning** | Statutory math, flight risk, outcome | 1,200 | 140 |
| **Adversarial** | Bias reversal, schema drift | 335 | 80 |

**Training mode:** Online — rewards computed via the live HF Space environment API.

In [ ]:
# Cell 1: Install dependencies
!pip install -q unsloth trl datasets wandb matplotlib
# For Kaggle: unsloth may need specific install
# !pip install unsloth[kaggle]

In [ ]:
# Cell 2: Clone repo and verify data
import os

REPO = "https://huggingface.co/spaces/Faiz-UndertriAI/Undertrial"
if not os.path.exists("data/episodes"):
    !git clone {REPO} undertrial_repo
    os.chdir("undertrial_repo")

# Verify episodes exist
import glob
episode_files = glob.glob("data/episodes/episodes_stage_*.jsonl")
print(f"Found {len(episode_files)} episode files")
for f in sorted(episode_files):
    lines = sum(1 for _ in open(f))
    print(f"  {f}: {lines} episodes")

In [ ]:
# Cell 3: Configure environment URL
# Point to your deployed HF Space for online reward computation
ENV_URL = "https://draken1606-undertrial-ai.hf.space"  # <-- UPDATE THIS

# Quick health check
import urllib.request, json
try:
    resp = urllib.request.urlopen(f"{ENV_URL}/health", timeout=10)
    health = json.loads(resp.read())
    print(f"\u2705 Environment is live: {health}")
except Exception as e:
    print(f"\u26a0\ufe0f  Cannot reach {ENV_URL}: {e}")
    print("Falling back to offline mode (set ENV_URL = None below)")
    ENV_URL = None

In [ ]:
# Cell 4: Import training functions (single source of truth)
import sys
sys.path.insert(0, '.')

from training.train_grpo import (
    load_episodes,
    combined_reward,
    train_curriculum,
    evaluate_on_stage,
    save_training_plots,
    save_comparison_plot,
    DIFFICULTY_MAP,
    DIFFICULTY_NAMES,
)
print("[OK] Training functions imported from train_grpo.py")
print(f"Difficulty levels: {list(DIFFICULTY_MAP.keys())}")
for k, v in DIFFICULTY_MAP.items():
    print(f"  {k}: stages={v['stages']}, sample={v['sample']}, steps={v['steps']}")

In [ ]:
# Cell 5: Verify episode loading + reward fixes
for diff in ['format', 'reasoning', 'adversarial']:
    eps = load_episodes('./data/episodes', difficulty=diff)
    granted = sum(1 for e in eps if 'grant' in e['ground_truth']['outcome'].lower())
    print(f"{diff:12s}: {len(eps)} episodes ({granted} granted, {len(eps)-granted} denied)")

# Verify reward fixes are active
from server.reward import compute_outcome_match, compute_flight_risk_accuracy
import json
ep = json.loads(open('data/episodes/episodes_stage_1.jsonl').readline())
gt = ep['ground_truth']
print(f"\nReward fix check:")
print(f"  Empty flight_risk = {compute_flight_risk_accuracy('', gt):.2f} (should be 0.00)")
print(f"  Wrong direction   = {compute_outcome_match('Bail Denied', gt):.2f} (should be -0.30)")

In [ ]:
# Cell 6: Full curriculum training (300 steps)
# Online mode: rewards via live HF Space (~6-8h on T4)
# Offline mode: in-process rewards (~2.5h on T4)
results = train_curriculum(
    episodes_dir="./data/episodes",
    output_dir="./output/undertrial_grpo",
    difficulties=["format", "reasoning", "adversarial"],
    model_name="unsloth/Qwen2.5-7B-Instruct",
    wandb_disabled=False,
    max_completion_length=384,
    env_url=ENV_URL,  # None = offline, URL = online
)

In [ ]:
# Cell 7: Display results and ALL reward curves
import json
from pathlib import Path
from IPython.display import Image, display

# Show results
results_path = Path("./output/undertrial_grpo/curriculum_results.json")
if results_path.exists():
    data = json.loads(results_path.read_text())
    print("=== CURRICULUM RESULTS ===")
    for level, r in data.get("levels", {}).items():
        print(f"  {level}: {r['baseline']:.4f} \u2192 {r['post']:.4f} (\u0394 = {r['delta']:+.4f})")

# Show combined all-levels reward curve
combined_path = Path("./output/undertrial_grpo/plots/reward_curve_all_levels.png")
if combined_path.exists():
    print("\n--- ALL LEVELS Reward Curve ---")
    display(Image(filename=str(combined_path)))

# Show per-level reward curves
for level in ['format', 'reasoning', 'adversarial']:
    plot_path = Path(f"./output/undertrial_grpo/plots/reward_curve_{level}.png")
    if plot_path.exists():
        print(f"\n--- {level.upper()} Reward Curve ---")
        display(Image(filename=str(plot_path)))

# Show comparison bar chart
cmp_path = Path("./output/undertrial_grpo/plots/before_after_comparison.png")
if cmp_path.exists():
    print("\n--- Before vs After ---")
    display(Image(filename=str(cmp_path)))

# Also check Kaggle persistent path
kaggle_plots = Path("/kaggle/working/undertrial_plots")
if kaggle_plots.exists():
    print(f"\n\u2705 Plots also saved to {kaggle_plots}")
    for p in sorted(kaggle_plots.glob("*.png")):
        print(f"  {p.name}")

In [ ]:
# Cell 8: Safety net — explicitly save plots to /kaggle/working/
# Run this if you're worried about losing plots
import shutil
from pathlib import Path

src_dirs = [
    Path("./output/undertrial_grpo/plots"),
    Path("./output/undertrial_grpo/level_format/plots"),
    Path("./output/undertrial_grpo/level_reasoning/plots"),
    Path("./output/undertrial_grpo/level_adversarial/plots"),
]
dst = Path("/kaggle/working/undertrial_all_plots")
dst.mkdir(parents=True, exist_ok=True)

saved = 0
for src in src_dirs:
    if src.exists():
        for png in src.glob("*.png"):
            prefix = src.parent.name if src.parent.name != "undertrial_grpo" else "root"
            out_name = f"{prefix}_{png.name}"
            shutil.copy2(str(png), str(dst / out_name))
            saved += 1

# Also copy results JSON
results_json = Path("./output/undertrial_grpo/curriculum_results.json")
if results_json.exists():
    shutil.copy2(str(results_json), str(dst / "curriculum_results.json"))
    saved += 1

print(f"\u2705 Saved {saved} files to {dst}")
for f in sorted(dst.iterdir()):
    print(f"  {f.name} ({f.stat().st_size // 1024} KB)")

In [ ]:
# Cell 9: Merge LoRA + push to Hub (optional)
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="./output/undertrial_grpo/final",
    max_seq_length=2048,
)

# Save merged model locally
model.save_pretrained_merged(
    "./output/undertrial_grpo/merged",
    tokenizer,
    save_method="merged_16bit",
)
print("Merged model saved to ./output/undertrial_grpo/merged")

# Uncomment to push to Hub:
# model.push_to_hub_merged(
#     "your-username/undertrial-7b-grpo",
#     tokenizer,
#     save_method="merged_16bit",
# )